# 02 — Análises: perguntas do desafio e lentes por área

Pergunta central: **quais produtos, categorias e estratégias comerciais merecem maior atenção e quais oportunidades os dados revelam?**

Cadeia investigada: **Preço → Desconto → Engajamento → Satisfação**.

Premissas (ver `src/config.py` e `data/processed/log_tratamento.json`):
- *Engajamento* = `rating_count` (quantidade de avaliações). A base não tem vendas nem datas.
- Somas de engajamento contam **1 vez por grupo de variantes** (mesma marca + nota + contagem), porque na Amazon as variantes compartilham avaliações. Sem isso, o total ficaria 27% maior.
- *Quadrantes*: cortes na **mediana** de nota e de avaliações.
- Customer Insights: sentimento com VADER (reviews em inglês) e temas por palavra-chave. Cada produto tem no máximo 8 reviews, e elas não trazem nota individual.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
from src import analises as A
from src.config import DIR_TABELAS, DIR_PROCESSADOS
pd.set_option("display.max_colwidth", 70)
DIR_TABELAS.mkdir(parents=True, exist_ok=True)
v, r, t = A.carregar()
def salvar(df, nome):
    df.to_csv(DIR_TABELAS / f"{nome}.csv")
    return df
print(json.dumps(json.loads((DIR_PROCESSADOS / "log_tratamento.json").read_text()), indent=1, ensure_ascii=False))

{
 "linhas_raw": 1465,
 "produtos_unicos_raw": 1351,
 "rating_invalido": 1,
 "qtd_avaliacoes_nula": 2,
 "duplicados_removidos": 114,
 "produtos_sem_avaliacao": 3,
 "produtos_em_grupo_de_variantes": 241,
 "grupos_de_avaliacao": 1203,
 "desconto_divergente_>1pp": 0,
 "produtos_final": 1351,
 "trechos_review": 10295,
 "trechos_com_tema": 5799
}


## Visão executiva

In [2]:
salvar(A.kpis(v).to_frame("valor"), "00_kpis")

,valor
produtos,1.348000e+03
avaliacoes_total_sem_variantes,1.869700e+07
nota_media,4.092000e+00
pct_produtos_abaixo_4,2.510000e-01
desconto_medio,4.670000e-01
preco_mediano_brl,5.391000e+01
pct_avaliacoes_top10pct_produtos,6.220000e-01


## 1. Quais categorias e produtos apresentam maior relevância?

In [3]:
salvar(A.por_categoria(v), "01_categorias")

,produtos,nota_media,pct_abaixo_4,avaliacoes_mediana,desconto_medio,avaliacoes_total,pct_avaliacoes
categoria,,,,,,,
Casa e Cozinha,447,4.041,0.322,2311.0,0.402,2990077.0,0.160
Outros,7,4.057,0.286,5985.0,0.431,118096.0,0.006
Eletrônicos,490,4.078,0.261,9129.5,0.499,10139250.0,0.542
Informática e Acessórios,373,4.155,0.174,7317.0,0.531,5299904.0,0.283
Escritório,31,4.310,0.000,4426.0,0.124,149675.0,0.008


In [4]:
salvar(A.por_categoria(v, "subcategoria"), "01b_subcategorias")

,produtos,nota_media,pct_abaixo_4,avaliacoes_mediana,desconto_medio,avaliacoes_total,pct_avaliacoes
subcategoria,,,,,,,
"Headphones,Earbuds & Accessories",63,3.921,0.524,37817.0,0.595,4153154.0,0.222
"Heating,Cooling & Air Quality",116,3.991,0.379,1743.5,0.407,551371.0,0.029
Wearable Technology,62,4.008,0.290,17161.5,0.697,860975.0,0.046
Home Storage & Organization,16,4.025,0.250,2366.5,0.461,53765.0,0.003
Kitchen & Home Appliances,307,4.053,0.309,2311.0,0.403,2051164.0,0.110
Home Audio,16,4.069,0.375,8746.5,0.497,293704.0,0.016
"Home Theater,TV & Video",156,4.071,0.282,1532.5,0.494,1510167.0,0.081
Cameras & Photography,16,4.125,0.312,11865.5,0.476,322657.0,0.017
Mobiles & Accessories,151,4.126,0.139,13246.0,0.415,2118920.0,0.113


## 2–3. Engajamento × satisfação: os quadrantes
Responde a *alto engajamento com baixa avaliação* (**Risco**) e a *boa avaliação com pouca visibilidade* (**Oportunidade**).

In [5]:
salvar(A.quadrantes(v), "02_quadrantes")

,produtos,nota_media,pct_abaixo_4,avaliacoes_mediana,desconto_medio,pct_avaliacoes
quadrante,,,,,,
Estrela,478,4.261,0.000,18066.5,0.424,0.747
Oportunidade,372,4.268,0.000,1465.5,0.470,0.031
Revisar,301,3.750,0.754,828.0,0.521,0.019
Risco,197,3.873,0.569,14062.0,0.478,0.203


In [6]:
salvar(A.lista_quadrante(v, "Risco"), "02b_top_risco")

,product_name,categoria,subcategoria,rating,qtd_avaliacoes,desconto_pct
17,boAt Airdopes 121v2 in-Ear True Wireless Earbuds with Upto 14 Hour...,Eletrônicos,"Headphones,Earbuds & Accessories",3.8,180998.0,0.57
31,boAt Airdopes 141 Bluetooth Truly Wireless in Ear Earbuds with mic...,Eletrônicos,"Headphones,Earbuds & Accessories",3.9,136954.0,0.67
32,"Nokia 105 Plus Single SIM, Keypad Mobile Phone with Wireless FM Ra...",Eletrônicos,Mobiles & Accessories,4.0,128311.0,0.22
36,Pigeon by Stovekraft Amaze Plus Electric Kettle (14289) with Stain...,Casa e Cozinha,Kitchen & Home Appliances,3.9,123365.0,0.48
42,Boult Audio Probass Curve Bluetooth Wireless in Ear Earphones with...,Eletrônicos,"Headphones,Earbuds & Accessories",3.8,103052.0,0.80
51,"JBL C50HI, Wired in Ear Headphones with Mic, One Button Multi-Func...",Eletrônicos,"Headphones,Earbuds & Accessories",3.9,92995.0,0.50
55,Noise ColorFit Pro 2 Full Touch Control Smart Watch with 35g Weigh...,Eletrônicos,Wearable Technology,4.0,92588.0,0.70
59,"JBL Tune 215BT, 16 Hrs Playtime with Quick Charge, in Ear Bluetoot...",Eletrônicos,"Headphones,Earbuds & Accessories",3.7,87798.0,0.50
60,"PTron Tangent Lite Bluetooth 5.0 Earphones with Mic, Hi-Fi Stereo ...",Eletrônicos,"Headphones,Earbuds & Accessories",3.5,83996.0,0.67
62,Boult Audio BassBuds X1 in-Ear Wired Earphones with 10mm Extra Bas...,Eletrônicos,"Headphones,Earbuds & Accessories",3.9,77027.0,0.67


In [7]:
salvar(A.lista_quadrante(v, "Oportunidade"), "02c_top_oportunidade")

,product_name,categoria,subcategoria,rating,qtd_avaliacoes,desconto_pct
723,"Instant Pot Air Fryer, Vortex 2QT, Touch Control Panel, 360° EvenC...",Casa e Cozinha,Kitchen & Home Appliances,4.8,3964.0,0.75
1306,Zuvexa USB Rechargeable Electric Foam Maker - Handheld Milk Wand M...,Casa e Cozinha,Kitchen & Home Appliances,4.7,54.0,0.62
924,"FIGMENT Handheld Milk Frother Rechargeable, 3-Speed Electric Froth...",Casa e Cozinha,Kitchen & Home Appliances,4.7,1729.0,0.56
859,Multifunctional 2 in 1 Electric Egg Boiling Steamer Egg Frying Pan...,Casa e Cozinha,Kitchen & Home Appliances,4.7,2300.0,0.56
832,Campfire Spring Chef Prolix Instant Portable Water Heater Geyser 1...,Casa e Cozinha,"Heating,Cooling & Air Quality",4.7,2591.0,0.57
1288,"VRPRIME Lint Roller Lint Remover for Clothes, Pet | 360 Sheets Reu...",Casa e Cozinha,Kitchen & Home Appliances,4.6,79.0,0.50
1263,Aquadpure Copper + Mineral RO+UV+UF 10 to 12 Liter RO + UV + TDS A...,Casa e Cozinha,Kitchen & Home Appliances,4.6,124.0,0.80
1264,Lint Remover Woolen Clothes Lint Extractor Battery Lint Removing M...,Casa e Cozinha,Kitchen & Home Appliances,4.6,124.0,0.57
1213,"WeCool S5 Long Selfie Stick, with Large Reinforced Tripod Stand up...",Eletrônicos,Mobiles & Accessories,4.6,245.0,0.55
814,"Sujata Dynamix, Mixer Grinder, 900 Watts, 3 Jars (White)",Casa e Cozinha,Kitchen & Home Appliances,4.6,2751.0,0.24


## 4. Existe relação entre descontos e avaliações?

In [8]:
salvar(A.desconto_vs_nota(v), "03_desconto_vs_nota")

,produtos,nota_media,pct_abaixo_4,avaliacoes_mediana,desconto_medio
faixa_desconto,,,,,
"(-0.01, 0.2]",175,4.169,0.166,6736.0,0.099
"(0.2, 0.4]",329,4.126,0.198,7241.0,0.310
"(0.4, 0.6]",470,4.071,0.283,3027.0,0.511
"(0.6, 0.8]",324,4.065,0.281,4147.0,0.699
"(0.8, 1.0]",50,3.974,0.420,3811.5,0.860


## 5. Quais padrões aparecem nas avaliações? O que os clientes valorizam e criticam?

In [9]:
print("validação: correlação sentimento médio × nota do produto =", A.cx_validacao_sentimento(v, r))
print(r["sentimento"].value_counts(normalize=True).round(3).to_dict())
salvar(A.temas(r, t), "04_temas")

validação: correlação sentimento médio × nota do produto = 0.326
{'Positivo': 0.769, 'Neutro': 0.126, 'Negativo': 0.105}


sentimento,Negativo,Neutro,Positivo,mencoes,pct_trechos,pct_negativo
tema,,,,,,
Qualidade e acabamento,225,113,1682,2020,0.196,0.111
Custo-benefício,117,88,1781,1986,0.193,0.059
Bateria e carregamento,139,146,964,1249,0.121,0.111
Instalação e uso,105,43,820,968,0.094,0.108
Desempenho e velocidade,75,113,779,967,0.094,0.078
Tamanho e encaixe,85,85,721,891,0.087,0.095
Som e áudio,87,35,563,685,0.067,0.127
Durabilidade e defeitos,186,64,380,630,0.061,0.295
Entrega e embalagem,103,29,411,543,0.053,0.190


---
# Lentes por área
## 💰 Pricing / Comercial

In [10]:
salvar(A.pricing_correlacoes(v), "10_pricing_correlacoes")

,qtd_avaliacoes,rating
desconto_pct,-0.135,-0.159
economia_brl,-0.004,-0.036
preco_desconto_brl,0.117,0.071


Dentro da mesma categoria folha, o efeito do desconto muda de sinal. Em eletrodomésticos, desconto alto acompanha nota baixa.

In [11]:
salvar(A.pricing_desconto_por_folha(v), "11_pricing_desconto_por_folha")

,produtos,rho_desconto_nota,rho_preco_avaliacoes
categoria_folha,,,
Instant Water Heaters,23,-0.50,0.34
Mixer Grinders,27,-0.50,0.25
Smart Televisions,38,-0.46,0.24
Hand Blenders,19,-0.38,0.45
In-Ear,49,-0.35,-0.21
Electric Heaters,20,-0.34,0.19
Lint Shavers,22,-0.32,0.37
Dry Irons,24,-0.31,0.01
Smartphones,32,-0.30,0.03


In [12]:
salvar(A.pricing_preco_final_9(v), "12_pricing_final_9")

,produtos,nota_media,pct_abaixo_4,avaliacoes_mediana,desconto_medio
preco_termina_em_9,,,,,
False,385,4.118,0.210,3234.0,0.412
True,963,4.082,0.268,6088.0,0.488


Hipótese testada e **não confirmada**: preço original inflado para simular desconto.

In [13]:
salvar(A.pricing_ancora_inflada(v), "13_pricing_ancora")

,produtos,nota_media,pct_abaixo_4,avaliacoes_mediana,desconto_medio
suspeita_ancora,,,,,
False,905,4.074,0.270,4744.0,0.485
True,19,4.068,0.316,4145.0,0.799


## 📣 Marketing / Conteúdo

In [14]:
mk = A.marketing_texto_anuncio(v)
salvar(mk["tamanho_titulo"], "20_mkt_titulo")

,produtos,nota_media,pct_abaixo_4,avaliacoes_mediana,desconto_medio
tamanho_titulo,,,,,
Q1 curto,349,4.148,0.175,6550.0,0.363
Q2,334,4.088,0.251,5367.5,0.458
Q3,329,4.055,0.298,3527.0,0.521
Q4 longo,336,4.074,0.286,3676.0,0.529


In [15]:
salvar(mk["tamanho_descricao"], "21_mkt_descricao")

,produtos,nota_media,pct_abaixo_4,avaliacoes_mediana,desconto_medio
tamanho_descricao,,,,,
Q1 curto,338,4.095,0.266,3811.0,0.401
Q2,337,4.094,0.228,6558.0,0.422
Q3,337,4.082,0.285,7333.0,0.517
Q4 longo,336,4.096,0.226,3528.5,0.527


## 📦 Compras / Fornecedores

In [16]:
salvar(A.compras_marcas(v), "30_compras_marcas")

,produtos,nota_media,pct_abaixo_4,avaliacoes_mediana,desconto_medio,pct_avaliacoes
marca,,,,,,
7seven,12,3.642,0.833,240.5,0.541,0.000
ptron,15,3.747,0.867,4415.0,0.802,0.011
boult,11,3.773,1.000,13049.0,0.765,0.016
lifelong,12,3.842,0.833,1737.0,0.519,0.004
zebronics,31,3.842,0.677,6742.0,0.479,0.019
pigeon,11,3.891,0.727,9791.0,0.447,0.027
prestige,14,3.950,0.571,14548.0,0.306,0.015
kent,14,3.957,0.429,2360.0,0.301,0.004
noise,20,3.965,0.350,11570.5,0.566,0.016


In [17]:
salvar(A.compras_espacos_vazios(v), "31_compras_espacos_vazios")

produtos  \
categoria                categoria_folha                                          
Eletrônicos              Screen Protectors                                    7   
Casa e Cozinha           Air Fryers                                           5   
                         Paints                                               3   
Eletrônicos              Disposable Batteries                                 7   
Informática e Acessórios External Hard Disks                                  6   
                         Internal Solid State Drives                          3   
Escritório               Wirebound Notebooks                                  4   
Eletrônicos              General Purpose Batteries & Battery Chargers         4   
Informática e Acessórios Mouse Pads                                           8   
                         Gaming Mice                                          6   
Escritório               Composition Notebooks                                7   
Informática e Acessórios Gaming Keyboards                                     3   
Eletrônicos              Micro SD                                            10   
Casa e Cozinha           Pop-up Toasters                                      3   
Informática e Acessórios Hard Disk Bags                                       3   
                         Notebook Computer Stands                             3   

                                                                       nota_media  \
categoria                categoria_folha                                            
Eletrônicos              Screen Protectors                                   4.47   
Casa e Cozinha           Air Fryers                                          4.46   
                         Paints                                              4.43   
Eletrônicos              Disposable Batteries                                4.41   
Informática e Acessórios External Hard Disks                                 4.40   
                         Internal Solid State Drives                         4.40   
Escritório               Wirebound Notebooks                                 4.40   
Eletrônicos              General Purpose Batteries & Battery Chargers        4.38   
Informática e Acessórios Mouse Pads                                          4.38   
                         Gaming Mice                                         4.38   
Escritório               Composition Notebooks                               4.37   
Informática e Acessórios Gaming Keyboards                                    4.37   
Eletrônicos              Micro SD                                            4.36   
Casa e Cozinha           Pop-up Toasters                                     4.33   
Informática e Acessórios Hard Disk Bags                                      4.33   
                         Notebook Computer Stands                            4.30   

                                                                       avaliacoes_mediana  
categoria                categoria_folha                                                   
Eletrônicos              Screen Protectors                                         6129.0  
Casa e Cozinha           Air Fryers                                                2981.0  
                         Paints                                                    7203.0  
Eletrônicos              Disposable Batteries                                      2866.0  
Informática e Acessórios External Hard Disks                                      37639.5  
                         Internal Solid State Drives                               3029.0  
Escritório               Wirebound Notebooks                                       7577.5  
Eletrônicos              General Purpose Batteries & Battery Chargers             13240.5  
Informática e Acessórios Mouse Pads                                                3336.0  
                         Gaming M

## ⚖️ Qualidade / Jurídico
Sinais de texto são **triagem** (há falso positivo) e devem orientar leitura manual, não ser lidos como taxa exata.

In [18]:
salvar(A.risco_sinais_texto(v, r), "40_risco_sinais")

,trechos,produtos,pct_negativo,pct_negativo_base,principais_categorias
sinal,,,,,
falsificacao,40,36,0.325,0.105,"Micro SD, USBCables, Remote Controls"
seguranca,55,49,0.109,0.105,"Smartphones, Electric Kettles, Mixer Grinders"
presente,58,56,0.034,0.105,"Smartphones, Smart Watches, USBCables"
concorrente,181,165,0.133,0.105,"USBCables, Smartphones, Smart Televisions"


In [19]:
salvar(A.risco_nota_vs_texto(v, r), "41_risco_nota_vs_texto")

,product_name,categoria_folha,rating,pct_trechos_negativos,qtd_avaliacoes
29,Samsung EVO Plus 64GB microSDXC UHS-I U1 130MB/s Full HD & 4K UHD ...,Micro SD,4.3,0.375,140036.0
123,"Bajaj Rex 500W Mixer Grinder with Nutri-Pro Feature, 3 Jars, White",Mixer Grinders,4.2,0.375,41349.0
143,"Logitech M221 Wireless Mouse, Silent Buttons, 2.4 GHz with USB Min...",Mice,4.4,0.375,34852.0
332,Samsung 80 cm (32 Inches) Wondertainment Series HD Ready LED Smart...,Smart Televisions,4.3,0.429,16299.0
516,Cuzor 12V Mini ups for WiFi Router | Power Backup up to 4 Hours | ...,Routers,4.5,0.375,8656.0
531,"HP X200 Wireless Mouse with 2.4 GHz Wireless connectivity, Adjusta...",Mice,4.2,0.375,8258.0
569,OnePlus 108 cm (43 inches) Y Series 4K Ultra HD Smart Android LED ...,Smart Televisions,4.2,0.444,7298.0
672,"Samsung Galaxy Watch4 Bluetooth(4.4 cm, Black, Compatible with And...",Smart Watches,4.3,0.455,4744.0
676,HP K500F Backlit Membrane Wired Gaming Keyboard with Mixed Color L...,Gaming Keyboards,4.3,0.429,4723.0
720,VW 60 cm (24 inches) Premium Series HD Ready LED TV VW24A (Black),Standard Televisions,4.2,0.500,4003.0


## 🛠️ Pós-venda / CX

In [20]:
salvar(A.cx_tempo_ate_falha(v, r), "50_cx_tempo_ate_falha")

,mencoes,pct
dias_ate_falha,,
até 1 semana,29,0.232
até 1 mês,32,0.256
1-3 meses,22,0.176
3-6 meses,12,0.096
6-12 meses,24,0.192
mais de 1 ano,6,0.048


In [21]:
salvar(A.cx_temas_por(v, r, t, "faixa_preco"), "51_cx_temas_por_preco")

faixa_preco,Q1 barato,Q2,Q3,Q4 caro
tema,,,,
Bateria e carregamento,0.125,0.105,0.126,0.128
Custo-benefício,0.184,0.180,0.174,0.234
Desempenho e velocidade,0.081,0.081,0.092,0.122
Durabilidade e defeitos,0.065,0.065,0.063,0.052
Entrega e embalagem,0.048,0.049,0.053,0.061
Garantia e atendimento,0.027,0.043,0.051,0.058
Instalação e uso,0.063,0.079,0.082,0.153
Qualidade e acabamento,0.200,0.207,0.165,0.212
Som e áudio,0.016,0.074,0.083,0.097


In [22]:
salvar(A.cx_negatividade_tema_por_desconto(v, r, t), "52_cx_negatividade_por_desconto")

faixa,até 40%,40-60%,acima de 60%
tema,,,
Bateria e carregamento,0.095,0.150,0.096
Custo-benefício,0.055,0.075,0.044
Desempenho e velocidade,0.084,0.079,0.067
Durabilidade e defeitos,0.363,0.272,0.258
Entrega e embalagem,0.209,0.169,0.188
Garantia e atendimento,0.239,0.207,0.152
Instalação e uso,0.107,0.105,0.120
Qualidade e acabamento,0.108,0.125,0.100
Som e áudio,0.137,0.129,0.111
